# 10 — Krahasimi i modeleve, pasiguria dhe metodologjia

## Konteksti dhe problemi shkencor

Një dataset i vetëm i ngjarjeve fizike përdoret për të krahasuar drejt pesë familje modelesh, duke përfshirë variabilitetin, robustësinë dhe interpretueshmërinë.

### Objektivat e të nxënit

- të bëni krahasim me të njëjtat fold-e;
- të raportoni variabilitet dhe bootstrap;
- të testoni robustësinë;
- të zgjidhni modelin mbi kritere shkencore.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Një dataset, pesë modele

In [ ]:
df=pd.read_csv(DATA/'physics_model_comparison.csv'); X=df.drop(columns='signal'); y=df.signal; df.head()

## 2. Test final i paprekur

In [ ]:
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,stratify=y,random_state=2026); print(len(Xdev),len(Xtest))

## 3. Modelet dhe pipelines

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
models={'LogReg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000)),'kNN':make_pipeline(StandardScaler(),KNeighborsClassifier(11)),'Tree':DecisionTreeClassifier(max_depth=5,random_state=0),'RF':RandomForestClassifier(n_estimators=250,random_state=0),'SVM':make_pipeline(StandardScaler(),SVC(probability=True,C=3))}

## 4. Cross-validation me të njëjtat fold-e

In [ ]:
from sklearn.metrics import make_scorer,roc_auc_score
cv=StratifiedKFold(8,shuffle=True,random_state=1); rows=[]
for name,m in models.items():
 r=cross_validate(m,Xdev,ydev,cv=cv,scoring=['accuracy','roc_auc'],return_train_score=True); rows.append({'model':name,'train_acc':r['train_accuracy'].mean(),'cv_acc':r['test_accuracy'].mean(),'cv_acc_sd':r['test_accuracy'].std(),'cv_auc':r['test_roc_auc'].mean(),'fit_s':r['fit_time'].mean()})
res=pd.DataFrame(rows).set_index('model'); res

## 5. Nuk ka vetëm një fitues

In [ ]:
res[['cv_acc','cv_acc_sd','cv_auc','fit_s']].plot.bar(figsize=(9,4)); plt.show()

## 6. Ndjeshmëria ndaj scaling

In [ ]:
for k in [3,11,31]: print(k,cross_val_score(KNeighborsClassifier(k),Xdev,ydev,cv=cv).mean(),cross_val_score(make_pipeline(StandardScaler(),KNeighborsClassifier(k)),Xdev,ydev,cv=cv).mean())

## 7. Robustësia ndaj zhurmës

In [ ]:
rng=np.random.default_rng(5); scales=[0,.1,.3,.6]; robust=[]
for s in scales:
 Xn=Xdev.copy(); Xn.iloc[:,:3]=Xn.iloc[:,:3]+rng.normal(0,s,Xn.iloc[:,:3].shape)*Xn.iloc[:,:3].std().to_numpy();
 for name,m in models.items(): robust.append((s,name,cross_val_score(m,Xn,ydev,cv=4).mean()))
rb=pd.DataFrame(robust,columns=['noise','model','score']); rb.pivot(index='noise',columns='model',values='score').plot(marker='o'); plt.show()

## 8. Interpretueshmëria

In [ ]:
lr=models['LogReg'].fit(Xdev,ydev); coef=pd.Series(lr[-1].coef_[0],index=X.columns); print(coef.sort_values()); rf=models['RF'].fit(Xdev,ydev); print(pd.Series(rf.feature_importances_,index=X.columns).sort_values())

## 9. Bootstrap i AUC në test final

In [ ]:
best=models['LogReg'].fit(Xdev,ydev); prob=best.predict_proba(Xtest)[:,1]; rng=np.random.default_rng(0); aucs=[]
for _ in range(1000):
 idx=rng.integers(0,len(ytest),len(ytest)); yt=ytest.to_numpy()[idx]; pr=prob[idx]
 if len(np.unique(yt))==2: aucs.append(roc_auc_score(yt,pr))
print(np.mean(aucs),np.percentile(aucs,[2.5,97.5]))

## 10. Data leakage demonstrim

In [ ]:
# Përzgjedhja e një veçorie zhurme sipas të gjithë dataset-it do të ishte metodologjikisht e gabuar.
print('Kolona noise_feature është futur qëllimisht për kontroll interpretimi.')

## 11. Testi final

In [ ]:
for name,m in models.items(): m.fit(Xdev,ydev); print(name,m.score(Xtest,ytest))

## 12. Ushtrime

- Kryeni tuning të secilit model me nested CV.
- Matni permutation importance.
- Ndryshoni klasën pozitive në 20% dhe përsëritni analizën.
- Shkruani një faqe ku zgjidhni modelin duke peshuar interpretueshmëri, stabilitet dhe performancë.

## 13. Shembull i plotë

Ky notebook është vetë shembulli i integruar. Plotësoni një tabelë vendimmarrjeje me kolonat: performancë, variabilitet, kosto, scaling, interpretueshmëri, robustësi.

## 14. Playground

Shtoni një veçori të re fizikisht informative dhe një tjetër zhurme. A i dallojnë modelet? Si ndryshojnë importance dhe koeficientët?